In [1]:
!pip install roboflow -q

from pathlib import Path
from dotenv import load_dotenv
import os

# Load .env from the project root
load_dotenv(Path('.env'))

RF_API_KEY = os.getenv('RF_API_KEY')
RF_WORKSPACE = os.getenv('RF_WORKSPACE')
RF_PROJECT = os.getenv('RF_PROJECT')
RF_VERSION = int(os.getenv('RF_VERSION', '1'))


from roboflow import Roboflow

rf = Roboflow(api_key=RF_API_KEY)
version = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)
dataset = version.download('yolov8')

DATASET_DIR = Path(dataset.location)
print(f'Downloaded to: {DATASET_DIR}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 38.1 MB/s eta 0:00:00
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to HairNet-Compliance-Detection-1 in yolov8:: 100%|██████████| 851/851 [00:00<00:00, 1251.64it/s]

Downloaded to: /content/HairNet-Compliance-Detection-1


# HairNet Compliance Detection

### MLflow Training

Each run varies:
- Model size (`yolov26n`, `yolov26m`, `yolo12n`, `yolo12m`)
- Epochs
- Image size
- Augmentation settings

At the end, MLflow lets us compare all runs and pick the winner.

## 0. Install Dependencies

In [2]:
!pip install ultralytics mlflow -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 74.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 74.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 887.6/

## 1. Imports & Setup

In [3]:
import os
import sys
from pathlib import Path

import mlflow
import mlflow.artifacts
import pandas as pd
from ultralytics import YOLO


DATA_YAML = Path('/content/HairNet-Compliance-Detection-1/data.yaml')
assert DATA_YAML.exists(), f'data.yaml not found at {DATA_YAML}. Run the download cell first.'

# MLflow tracking

MLFLOW_TRACKING_URI = 'mlruns'
EXPERIMENT_NAME     = 'hairnet-compliance-detection'

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

print(f'MLflow tracking URI : {mlflow.get_tracking_uri()}')
print(f'Experiment          : {EXPERIMENT_NAME}')
print(f'Data YAML           : {DATA_YAML.resolve()}')

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
2026/05/13 17:43:20 INFO mlflow.tracking.fluent: Experiment with name 'hairnet-compliance-detection' does not exist. Creating a new experiment.


MLflow tracking URI : mlruns
Experiment          : hairnet-compliance-detection
Data YAML           : /content/HairNet-Compliance-Detection-1/data.yaml


## 2. Define Experiment Configurations

Each dict is one MLflow run. Add or remove entries to control what gets trained.

#### [yolo AUG doc] https://docs.ultralytics.com/guides/yolo-data-augmentation

In [4]:
AUG_OFF = dict(hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, translate=0.0,
               scale=0.0, fliplr=0.0, mosaic=0.0, erasing=0.0, auto_augment=None)

AUG_ON = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, translate=0.1, scale=0.5,
              fliplr=0.5, flipud=0.1, mosaic=1.0,
              erasing=0.4, degrees=10.0, auto_augment='randaugment')

RUNS = [
    # ── Baselines (no augmentation) ───────────────────────────────────
    {'run_name': 'yolov26n-baseline', 'model': 'yolo26n.pt', 'epochs': 30, 'batch': 16, **AUG_OFF},
    {'run_name': 'yolov26m-baseline', 'model': 'yolo26m.pt', 'epochs': 30, 'batch': 8,  **AUG_OFF},
    {'run_name': 'yolo12n-baseline',  'model': 'yolo12n.pt',  'epochs': 30, 'batch': 16, **AUG_OFF},
    {'run_name': 'yolo12m-baseline',  'model': 'yolo12m.pt',  'epochs': 30, 'batch': 8,  **AUG_OFF},

    # ── Augmented ─────────────────────────────────────────────────────
    {'run_name': 'yolov26n-augmented', 'model': 'yolo26n.pt', 'epochs': 50, 'batch': 16, **AUG_ON},
    {'run_name': 'yolov26m-augmented', 'model': 'yolo26m.pt', 'epochs': 50, 'batch': 8,  **AUG_ON},
    {'run_name': 'yolo12n-augmented',  'model': 'yolo12n.pt',  'epochs': 50, 'batch': 16, **AUG_ON},
    {'run_name': 'yolo12m-augmented',  'model': 'yolo12m.pt',  'epochs': 50, 'batch': 8,  **AUG_ON},
]

# shared params applied to every run
SHARED = dict(imgsz=640, optimizer='auto', patience=10)

print(f'{len(RUNS)} runs defined:\n')
print(f"  {'Run':<25} {'Model':<14} {'Epochs':>6} {'Batch':>6} {'Aug'}")
print('  ' + '-' * 60)
for r in RUNS:
    aug = 'ON' if r['mosaic'] > 0 else 'OFF'
    print(f"  {r['run_name']:<25} {r['model']:<14} {r['epochs']:>6} {r['batch']:>6}   {aug}")

8 runs defined:

  Run                       Model          Epochs  Batch Aug
  ------------------------------------------------------------
  yolov26n-baseline         yolo26n.pt         30     16   OFF
  yolov26m-baseline         yolo26m.pt         30      8   OFF
  yolo12n-baseline          yolo12n.pt         30     16   OFF
  yolo12m-baseline          yolo12m.pt         30      8   OFF
  yolov26n-augmented        yolo26n.pt         50     16   ON
  yolov26m-augmented        yolo26m.pt         50      8   ON
  yolo12n-augmented         yolo12n.pt         50     16   ON
  yolo12m-augmented         yolo12m.pt         50      8   ON


In [5]:
print(RUNS)

[{'run_name': 'yolov26n-baseline', 'model': 'yolo26n.pt', 'epochs': 30, 'batch': 16, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolov26m-baseline', 'model': 'yolo26m.pt', 'epochs': 30, 'batch': 8, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolo12n-baseline', 'model': 'yolo12n.pt', 'epochs': 30, 'batch': 16, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolo12m-baseline', 'model': 'yolo12m.pt', 'epochs': 30, 'batch': 8, 'hsv_h': 0.0, 'hsv_s': 0.0, 'hsv_v': 0.0, 'translate': 0.0, 'scale': 0.0, 'fliplr': 0.0, 'mosaic': 0.0, 'erasing': 0.0, 'auto_augment': None}, {'run_name': 'yolov26n-augmented', 'model': 'yolo26n.pt', 'epochs': 50, 'batch': 16, 'hsv_h': 0

## 3. Training Loop

Each config gets its own MLflow run. Parameters, metrics, and the best weights are all logged automatically.

In [6]:
FIXED_KEYS = {'run_name', 'model', 'epochs', 'batch'}

for cfg in RUNS:
    run_name = cfg['run_name']
    aug_args = {k: v for k, v in cfg.items() if k not in FIXED_KEYS and v is not None}

    print(f'\n{"="*60}')
    print(f'Starting run: {run_name}')
    print(f'{"="*60}')

    mlflow.end_run()
    with mlflow.start_run(run_name=run_name):

        # ── Log all parameters
        mlflow.log_params({
            'model' : cfg['model'],
            'epochs': cfg['epochs'],
            'batch' : cfg['batch'],
            **SHARED,
            **aug_args,
        })

        # ── Train
        model   = YOLO(cfg['model'])
        results = model.train(
            data     = str(DATA_YAML),
            epochs   = cfg['epochs'],
            batch    = cfg['batch'],
            project  = 'runs/train',
            name     = run_name,
            exist_ok = True,
            verbose  = False,
            **SHARED,
            **aug_args,
        )

        # ── Log final metrics
        metrics = results.results_dict
        mlflow.log_metrics({
            'mAP50'    : metrics.get('metrics/mAP50(B)',     0),
            'mAP50-95' : metrics.get('metrics/mAP50-95(B)', 0),
            'precision': metrics.get('metrics/precision(B)', 0),
            'recall'   : metrics.get('metrics/recall(B)',    0),
            'box_loss' : metrics.get('val/box_loss',         0),
            'cls_loss' : metrics.get('val/cls_loss',         0),
            'fitness'  : results.fitness
        })

        # ── Per-epoch loss curves
        csv_path = Path(f'/content/runs/detect/runs/{run_name}/results.csv')
        if csv_path.exists():
            df_results = pd.read_csv(csv_path)
            df_results.columns = df_results.columns.str.strip()
            for _, row in df_results.iterrows():
                epoch = int(row['epoch'])
                mlflow.log_metrics({
                    'train/box_loss': row.get('train/box_loss', 0),
                    'train/cls_loss': row.get('train/cls_loss', 0),
                    'train/dfl_loss': row.get('train/dfl_loss', 0),
                    'val/box_loss'  : row.get('val/box_loss',   0),
                    'val/cls_loss'  : row.get('val/cls_loss',   0),
                    'val/dfl_loss'  : row.get('val/dfl_loss',   0),
                }, step=epoch)

        # ── Log best weights as artifact copy the best to mlflow storage
        best_weights = Path(f'/content/runs/detect/runs/train/{run_name}/weights/best.pt')
        if best_weights.exists():
            mlflow.log_artifact(str(best_weights), artifact_path='weights')

        print(f'Run complete — mAP50: {metrics.get("metrics/mAP50(B)", 0):.4f}')


Starting run: yolov26n-baseline
Ultralytics 8.4.50 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/HairNet-Compliance-Detection-1/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.0, exist_ok=True, fliplr=0.0, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.0, hsv_s=0.0, hsv_v=0.0, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n.pt, momentum=0.937, mosaic=0.0, multi_scale=0.0, name=yolov26n-baseline, nbs=64, nms=False, opset=None, op

2026/05/13 17:43:36 INFO mlflow.tracking.fluent: Experiment with name 'runs/train' does not exist. Creating a new experiment.
2026/05/13 17:43:36 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/13 17:43:36 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(0a2abe883e56454e8a687b66e8e35687) to mlruns
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri mlruns'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/runs/detect/runs/train/yolov26n-baseline
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/30      2.54G      1.324      4.047   0.005885        172        640: 100% ━━━━━━━━━━━━ 18/18 2.1s/it 36.9s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 3.1s/it 9.3s
                   all         93       1542     0.0182      0.282      0.139     0.0944

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      2.67G      1.023       2.89    0.00422        168        640: 100% ━━━━━━━━━━━━ 18/18 2.5it/s 7.1s
                 Class     Ima

2026/05/13 17:49:24 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/13 17:49:24 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(36f05d43a269474b8fa0f50958d53608) to mlruns
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri mlruns'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/runs/detect/runs/train/yolov26m-baseline
Starting training for 30 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/30      4.72G      1.092      2.683   0.004734         46        640: 100% ━━━━━━━━━━━━ 36/36 1.1it/s 32.6s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 1.3s/it 7.8s
                   all         93       1542      0.847      0.799      0.906      0.631

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/30      4.97G     0.8754     0.7978   0.003767         38        640: 100% ━━━━━━━━━━━━ 36/36 2.5it/s 14.6s
                 Class     Im

...

MLflow: logging run_id(45f9ef797c374b53990b8fb085e43185) to mlruns
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri mlruns'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/runs/detect/runs/train/yolov26m-augmented
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      4.79G      1.537      2.999   0.006597         76        640: 100% ━━━━━━━━━━━━ 36/36 2.0it/s 18.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 3.2it/s 1.9s
                   all         93       1542      0.751      0.739      0.795      0.454

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      5.04G      1.414      1.329   0.005979         40        640: 100% ━━━━━━━━━━━━ 36/36 2.3it/s 15.4s
                 Class     I

2026/05/13 18:50:50 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/13 18:50:50 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(0d60f06bfd9a4a1a9149171436ff6a36) to mlruns
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri mlruns'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/runs/detect/runs/train/yolo12n-augmented
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      4.02G      1.799      3.559      1.394        203        640: 100% ━━━━━━━━━━━━ 18/18 1.1it/s 16.5s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 1.9it/s 1.6s
                   all         93       1542     0.0231      0.325      0.196     0.0858

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      4.03G      1.511      2.504      1.182        215        640: 100% ━━━━━━━━━━━━ 18/18 2.0it/s 9.1s
                 Class     Ima

2026/05/13 19:01:49 WARNING mlflow.spark: With Pyspark >= 3.2, PYSPARK_PIN_THREAD environment variable must be set to false for Spark datasource autologging to work.
2026/05/13 19:01:49 INFO mlflow.tracking.fluent: Autologging successfully enabled for pyspark.


MLflow: logging run_id(2226196a7be04073907bd58474fa1189) to mlruns
MLflow: view at http://127.0.0.1:5000 with 'mlflow server --backend-store-uri mlruns'
MLflow: disable with 'yolo settings mlflow=False'
Image sizes 640 train, 640 val
Using 2 dataloader workers
Logging results to /content/runs/detect/runs/train/yolo12m-augmented
Starting training for 50 epochs...

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/50      6.14G      1.526      1.694       1.33         76        640: 100% ━━━━━━━━━━━━ 36/36 1.7it/s 20.7s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 6/6 2.6it/s 2.3s
                   all         93       1542      0.286      0.683      0.305       0.16

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/50      6.39G      1.346      1.004       1.25         40        640: 100% ━━━━━━━━━━━━ 36/36 2.0it/s 18.0s
                 Class     Im

In [7]:
from google.colab import files
import shutil

shutil.make_archive('runs', 'zip', '/content/runs')
files.download('runs.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
shutil.make_archive('mlruns', 'zip', '/content/mlruns')
files.download('mlruns.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>